# PyTorch Advanced — From Transformer Variants to Production

> This notebook picks up where `pytorch_zero_to_hero` left off. Every section builds on the base Transformer code, covers a real-world technique used in modern LLMs, and ends with hands-on exercises.

---

## Table of Contents
1. [Setup & Shared Utilities](#1)
2. [Decoder-Only Architecture (GPT-style)](#2)
3. [Encoder-Only Architecture (BERT-style)](#3)
4. [Flash Attention — Memory-Efficient Attention](#4)
5. [torch.compile — Kernel Fusion & Graph Optimization](#5)
6. [Distributed Training with DDP](#6)
7. [Memory Profiling & Leak Detection](#7)
8. [Training Monitoring with TensorBoard](#8)
9. [Gradient Checkpointing & Activation Recomputation](#9)
10. [KV-Cache & Fast Autoregressive Inference](#10)
11. [Model Quantization (INT8 / FP16 static)](#11)
12. [Advanced Optimization Recipes](#12)


---
<a id="1"></a>
## Section 1 — Setup & Shared Utilities

We re-implement the core building blocks here so this notebook is **self-contained**. The implementations are identical to `pytorch_zero_to_hero` but consolidated into a single setup cell for convenience.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math, time, os, copy
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device   : {DEVICE}")

# ── Shared hyper-parameters ───────────────────────────────────────────────────
VOCAB_SIZE  = 256
SEQ_LEN     = 64
BATCH_SIZE  = 8
D_MODEL     = 128
NUM_HEADS   = 8
N_LAYERS    = 4
D_FF        = D_MODEL * 4
DROPOUT     = 0.1


In [ ]:
# ── Core building-blocks (from zero_to_hero) ────────────────────────────────
def causal_mask(size: int, device=torch.device("cpu")) -> torch.Tensor:
    return torch.triu(torch.ones(size, size, dtype=torch.bool, device=device), diagonal=1).unsqueeze(0)


def scaled_dot_product_attention(Q, K, V, mask=None, dropout=0.0):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    if dropout > 0.0 and torch.is_grad_enabled():
        weights = F.dropout(weights, p=dropout)
    return weights @ V, weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.dropout = dropout
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def _split(self, x):
        B, S, _ = x.shape
        return x.view(B, S, self.num_heads, self.d_k).transpose(1, 2)

    def _merge(self, x):
        B, _, S, _ = x.shape
        return x.transpose(1, 2).contiguous().view(B, S, -1)

    def forward(self, q, k, v, mask=None):
        Q, K, V = self._split(self.W_q(q)), self._split(self.W_k(k)), self._split(self.W_v(v))
        if mask is not None and mask.dim() == 3:
            mask = mask.unsqueeze(1)
        out, w = scaled_dot_product_attention(Q, K, V, mask, self.dropout)
        return self.W_o(self._merge(out)), w


class PositionwiseFFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, act="gelu"):
        super().__init__()
        activation = nn.GELU() if act == "gelu" else nn.ReLU()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), activation,
                                  nn.Dropout(dropout), nn.Linear(d_ff, d_model))
    def forward(self, x): return self.net(x)


class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.scale = math.sqrt(d_model)
    def forward(self, x): return self.embed(x) * self.scale


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return self.dropout(x + self.pe[:, :x.size(1)])


class SequenceDataset(Dataset):
    def __init__(self, n, seq_len, vocab_size):
        self.data = torch.randint(0, vocab_size, (n, seq_len))
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        return self.data[i][:-1], self.data[i][1:]


print("Shared utilities loaded ✓")


---
<a id="2"></a>
## Section 2 — Decoder-Only Architecture (GPT-style)

A **decoder-only** Transformer removes the encoder entirely. Every layer is a **masked self-attention** block — the model can only look at previous positions (causal mask). This is the architecture behind GPT, GPT-2, GPT-3, LLaMA, Mistral, and Falcon.

```
Input tokens
    │
TokenEmbedding + PositionalEncoding
    │
┌───┴─────────────────────────────┐
│  GPTBlock × N                   │
│  ┌────────────────────────────┐ │
│  │ Pre-LN ──► Masked MHA ──►+│ │
│  │ Pre-LN ──► FFN        ──►+│ │
│  └────────────────────────────┘ │
└─────────────────────────────────┘
    │
Final LayerNorm ──► Linear(d_model, vocab) ──► logits
```

**Key differences from encoder-decoder:**
- No cross-attention
- Causal mask applied in self-attention at every layer
- A single embedding table shared between input and output projection (weight tying)
- Pre-norm (LayerNorm before sublayer) for training stability


In [ ]:
class GPTBlock(nn.Module):
    """
    Single GPT decoder block (Pre-LN, no cross-attention).
    Used in GPT-2, LLaMA (with RMSNorm + SwiGLU as drop-in replacements).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1    = nn.LayerNorm(d_model)
        self.attn     = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2    = nn.LayerNorm(d_model)
        self.ffn      = PositionwiseFFN(d_model, d_ff, dropout, act="gelu")
        self.drop     = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Pre-LN masked self-attention
        attn_out, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x), mask=mask)
        x = x + self.drop(attn_out)
        # Pre-LN FFN
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x


class GPT(nn.Module):
    """
    GPT-style decoder-only language model.

    Args:
        vocab_size  : vocabulary size
        d_model     : embedding dimension
        num_heads   : attention heads
        num_layers  : number of GPTBlocks
        d_ff        : FFN inner dimension
        max_len     : maximum context length
        dropout     : dropout probability
    """
    def __init__(self, vocab_size, d_model=256, num_heads=8, num_layers=6,
                 d_ff=1024, max_len=512, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        # Learnable positional embedding (GPT-2 style)
        self.pos_emb   = nn.Embedding(max_len, d_model)
        self.drop      = nn.Dropout(dropout)
        self.blocks    = nn.ModuleList([
            GPTBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.ln_final  = nn.LayerNorm(d_model)
        self.head      = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: shared embedding / unembedding matrix
        self.head.weight = self.token_emb.weight

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """
        idx : (batch, seq_len) — token indices
        Returns logits of shape (batch, seq_len, vocab_size)
        """
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device).unsqueeze(0)   # (1, T)

        x    = self.drop(self.token_emb(idx) + self.pos_emb(positions))
        mask = causal_mask(T, device=idx.device)

        for block in self.blocks:
            x = block(x, mask)

        x = self.ln_final(x)
        return self.head(x)   # (B, T, vocab_size)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=20, temperature=1.0, top_k=None):
        """
        Greedy / top-k sampling autoregressive generation.
        idx : (1, T) seed token sequence
        """
        self.eval()
        for _ in range(max_new_tokens):
            logits = self(idx)[:, -1, :] / temperature       # (1, vocab)
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, -1:]] = float("-inf")
            probs    = torch.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)  # (1, 1)
            idx      = torch.cat([idx, next_tok], dim=1)
        return idx


# ── Instantiate and run a forward pass ───────────────────────────────────────
gpt = GPT(
    vocab_size=VOCAB_SIZE, d_model=D_MODEL, num_heads=NUM_HEADS,
    num_layers=N_LAYERS, d_ff=D_FF, max_len=SEQ_LEN, dropout=DROPOUT,
).to(DEVICE)

tokens = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN - 1)).to(DEVICE)
logits = gpt(tokens)
print(f"GPT forward output  : {logits.shape}")   # (8, 63, 256)

params = sum(p.numel() for p in gpt.parameters())
print(f"GPT total params    : {params:,}")

# Autoregressive generation (5 new tokens)
seed = torch.randint(0, VOCAB_SIZE, (1, 8)).to(DEVICE)
generated = gpt.generate(seed, max_new_tokens=5, temperature=0.8, top_k=50)
print(f"Generated sequence  : {generated.shape}  → {generated[0].tolist()}")


### 🏋️ Exercises — Section 2: Decoder-Only (GPT-style)

**E2.1** Implement **nucleus (top-p) sampling** in `GPT.generate`: filter vocabulary to the smallest set whose cumulative probability exceeds `p`, then sample from it. Compare output diversity between `top_k=50` and `top_p=0.9`.

**E2.2** Replace `nn.LayerNorm` in `GPTBlock` with **RMSNorm** (`x / rms(x) * γ`) and verify the model still trains and produces the same output shapes.

**E2.3** Add a **context window length check** to `GPT.forward` that raises a descriptive error when `T > max_len`, rather than silently failing with an index error.

**E2.4** Write a `count_flops_per_token(model)` function that estimates the number of FLOPs required to process a single token during inference. Use the formula: for each linear layer of shape `(m, n)`, FLOPs ≈ `2mn` per token.


In [ ]:
# ── E2.1 — Top-p (nucleus) sampling ──────────────────────────────────────────
@torch.no_grad()
def gpt_generate_nucleus(model, idx, max_new_tokens=20, temperature=1.0, top_p=0.9):
    model.eval()
    for _ in range(max_new_tokens):
        logits = model(idx)[:, -1, :] / temperature
        probs  = torch.softmax(logits, dim=-1)
        # Sort probs descending
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative = torch.cumsum(sorted_probs, dim=-1)
        # YOUR CODE HERE: remove tokens once cumulative prob > top_p
        # sorted_probs[cumulative - sorted_probs > top_p] = 0.0
        # sorted_probs = sorted_probs / sorted_probs.sum()  # renormalize
        # next_tok = torch.gather(sorted_indices, -1, torch.multinomial(sorted_probs, 1))
        next_tok = sorted_indices[:, :1]   # placeholder — replace with sampling
        idx = torch.cat([idx, next_tok], dim=1)
    return idx


# ── E2.2 — RMSNorm replacement ────────────────────────────────────────────────
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.eps   = eps

    def forward(self, x):
        # YOUR CODE HERE
        # rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        # return x / rms * self.gamma
        return x   # placeholder

# Swap LayerNorm → RMSNorm in GPTBlock and verify
# class GPTBlockRMS(GPTBlock):
#     def __init__(self, *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.norm1 = RMSNorm(args[0])
#         self.norm2 = RMSNorm(args[0])

# ── E2.3 — Context window guard ───────────────────────────────────────────────
# YOUR CODE HERE: Add assertion to GPT.forward:
# assert T <= self.pos_emb.num_embeddings, (
#     f"Sequence length {T} exceeds model max_len {self.pos_emb.num_embeddings}"
# )

# ── E2.4 — FLOP estimation ────────────────────────────────────────────────────
def count_flops_per_token(model: nn.Module) -> int:
    """
    Approximate FLOPs per forward token (inference).
    For each nn.Linear(m, n): FLOPs ≈ 2 * m * n
    """
    total = 0
    for module in model.modules():
        if isinstance(module, nn.Linear):
            # YOUR CODE HERE
            pass   # total += 2 * module.in_features * module.out_features
    return total

flops = count_flops_per_token(gpt)
print(f"Approx FLOPs/token: {flops:,}")


---
<a id="3"></a>
## Section 3 — Encoder-Only Architecture (BERT-style)

A **BERT-style** model uses the **full (bidirectional) attention** — every position can attend to every other position. The model is pre-trained with two tasks:

1. **Masked Language Modeling (MLM):** Replace 15% of tokens with `[MASK]`, predict originals.
2. **Next Sentence Prediction (NSP):** Classify whether two sentences are consecutive *(less impactful, dropped in RoBERTa)*.

```
Input: [CLS] w₁ w₂ [MASK] w₄ [SEP]
            │
    TokenEmbedding + SegmentEmbedding + PositionalEmbedding
            │
    EncoderBlock × N  (full bidirectional attention)
            │
    [CLS] hidden state ──► Classification head
    All positions      ──► MLM head (predict masked tokens)
```

**Fine-tuning:** freeze or fine-tune the pre-trained encoder, add a task-specific head on top of `[CLS]`.


In [ ]:
# ── Special token IDs ────────────────────────────────────────────────────────
CLS_ID  = VOCAB_SIZE       # [CLS] — classification token (prepended)
SEP_ID  = VOCAB_SIZE + 1   # [SEP] — separator
MASK_ID = VOCAB_SIZE + 2   # [MASK] — masked token
BERT_VOCAB = VOCAB_SIZE + 3


class BERTEncoderBlock(nn.Module):
    """Standard encoder block — no causal mask, bidirectional."""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn   = PositionwiseFFN(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, padding_mask=None):
        # padding_mask shape: (B, 1, 1, S) — True = padding position to block
        n = self.norm1(x)
        x = x + self.drop(self.attn(n, n, n, mask=padding_mask)[0])
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x


class BERT(nn.Module):
    """
    BERT-style encoder-only Transformer.
    Outputs:
      - pooled_output : (B, d_model) — [CLS] token representation for classification
      - sequence_output: (B, S, d_model) — per-token representations for MLM / NER / QA
    """
    def __init__(self, vocab_size, d_model=128, num_heads=8,
                 num_layers=4, d_ff=512, max_len=512, dropout=0.1):
        super().__init__()
        self.token_emb   = nn.Embedding(vocab_size, d_model)
        self.pos_emb     = nn.Embedding(max_len, d_model)
        self.segment_emb = nn.Embedding(2, d_model)   # sentence A=0, B=1
        self.norm        = nn.LayerNorm(d_model)
        self.drop        = nn.Dropout(dropout)
        self.layers      = nn.ModuleList([
            BERTEncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        # [CLS] pooler — linear + tanh, standard BERT
        self.pooler      = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.Tanh()
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, segment_ids=None, padding_mask=None):
        """
        input_ids   : (B, S) — token indices (first token should be CLS_ID)
        segment_ids : (B, S) — 0 for sentence A, 1 for sentence B
        padding_mask: (B, S) bool — True = PAD position
        """
        B, S = input_ids.shape
        positions    = torch.arange(S, device=input_ids.device).unsqueeze(0)
        if segment_ids is None:
            segment_ids = torch.zeros_like(input_ids)

        x = self.token_emb(input_ids) + self.pos_emb(positions) + self.segment_emb(segment_ids)
        x = self.drop(self.norm(x))

        # Expand padding mask for attention: (B, S) → (B, 1, 1, S)
        attn_mask = None
        if padding_mask is not None:
            attn_mask = padding_mask.unsqueeze(1).unsqueeze(1)

        for layer in self.layers:
            x = layer(x, padding_mask=attn_mask)

        sequence_output = x                         # (B, S, d_model)
        pooled_output   = self.pooler(x[:, 0, :])   # (B, d_model) — [CLS]
        return sequence_output, pooled_output


# ── MLM Head ──────────────────────────────────────────────────────────────────
class MLMHead(nn.Module):
    """Predicts the original token at each masked position."""
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.dense = nn.Linear(d_model, d_model)
        self.act   = nn.GELU()
        self.norm  = nn.LayerNorm(d_model)
        self.proj  = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        return self.proj(self.norm(self.act(self.dense(x))))


# ── Full BERT pre-training model ──────────────────────────────────────────────
class BERTForMLM(nn.Module):
    def __init__(self, vocab_size, **kwargs):
        super().__init__()
        self.bert    = BERT(vocab_size, **kwargs)
        self.mlm_head = MLMHead(kwargs.get("d_model", 128), vocab_size)

    def forward(self, input_ids, segment_ids=None, padding_mask=None):
        seq_out, cls_out = self.bert(input_ids, segment_ids, padding_mask)
        mlm_logits = self.mlm_head(seq_out)   # (B, S, vocab_size)
        return mlm_logits, cls_out


# ── Quick test: Masked Language Modeling step ─────────────────────────────────
def make_mlm_batch(batch_size, seq_len, vocab_size, mask_prob=0.15):
    """Create a batch with randomly masked tokens."""
    input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
    labels    = input_ids.clone()
    # Prepend [CLS]
    cls_col   = torch.full((batch_size, 1), CLS_ID)
    input_ids = torch.cat([cls_col, input_ids[:, :-1]], dim=1)

    # Mask random positions (skip [CLS] at position 0)
    mask = (torch.rand(batch_size, seq_len) < mask_prob)
    mask[:, 0] = False   # never mask [CLS]
    labels[~mask] = -100  # cross-entropy ignores -100
    input_ids[mask] = MASK_ID
    return input_ids, labels, mask


bert_mlm = BERTForMLM(BERT_VOCAB, d_model=D_MODEL, num_heads=NUM_HEADS,
                       num_layers=N_LAYERS, d_ff=D_FF, max_len=SEQ_LEN).to(DEVICE)

input_ids, labels, mask = make_mlm_batch(BATCH_SIZE, SEQ_LEN, BERT_VOCAB)
input_ids, labels = input_ids.to(DEVICE), labels.to(DEVICE)

mlm_logits, cls_repr = bert_mlm(input_ids)
print(f"MLM logits shape  : {mlm_logits.shape}")   # (8, 64, vocab)
print(f"CLS repr shape    : {cls_repr.shape}")      # (8, 128)

# MLM loss — only over masked positions
mlm_loss = F.cross_entropy(
    mlm_logits.reshape(-1, BERT_VOCAB),
    labels.reshape(-1),
    ignore_index=-100
)
print(f"MLM loss          : {mlm_loss.item():.4f}")
print(f"BERT total params : {sum(p.numel() for p in bert_mlm.parameters()):,}")


### 🏋️ Exercises — Section 3: Encoder-Only (BERT-style)

**E3.1** Implement a **sequence classification head** on top of `BERT`: take `pooled_output` (the `[CLS]` vector) and pass it through `Dropout → Linear(d_model, num_classes)`. Fine-tune on a binary classification task with random labels.

**E3.2** Implement the **80/10/10 rule** for MLM masking: of the 15% selected tokens, replace 80% with `[MASK]`, 10% with a random token, and 10% keep the original. Show this improves compared to always replacing with `[MASK]`.

**E3.3** Add a **Next Sentence Prediction (NSP) head**: given two sequences concatenated with `[SEP]`, predict whether the second sentence follows the first (binary classification on `[CLS]`).

**E3.4** Build a **token classification head** (for NER/POS tagging): apply `Linear(d_model, num_labels)` to every position in `sequence_output`. Compute the cross-entropy loss over non-padding positions only.


In [ ]:
# ── E3.1 — Sequence classification head ──────────────────────────────────────
class BERTForSequenceClassification(nn.Module):
    def __init__(self, vocab_size, num_classes, dropout=0.1, **bert_kwargs):
        super().__init__()
        self.bert = BERT(vocab_size, **bert_kwargs)
        d_model   = bert_kwargs.get("d_model", 128)
        # YOUR CODE HERE
        # self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, num_classes))

    def forward(self, input_ids, segment_ids=None, padding_mask=None):
        _, cls_repr = self.bert(input_ids, segment_ids, padding_mask)
        # YOUR CODE HERE
        # return self.classifier(cls_repr)
        return cls_repr   # placeholder

# ── E3.2 — 80/10/10 masking ──────────────────────────────────────────────────
def make_mlm_batch_80_10_10(batch_size, seq_len, vocab_size, mask_prob=0.15):
    input_ids = torch.randint(2, vocab_size, (batch_size, seq_len))
    labels    = input_ids.clone()
    selected  = torch.rand(batch_size, seq_len) < mask_prob

    # YOUR CODE HERE:
    # r = torch.rand(batch_size, seq_len)
    # input_ids[selected & (r < 0.8)] = MASK_ID          # 80% → [MASK]
    # input_ids[selected & (r >= 0.8) & (r < 0.9)] = ... # 10% → random token
    # leave 10% unchanged
    # labels[~selected] = -100

    return input_ids, labels, selected

# ── E3.3 — NSP head ───────────────────────────────────────────────────────────
class NSPHead(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        # YOUR CODE HERE: binary classifier on [CLS]
        pass

    def forward(self, cls_repr):
        # YOUR CODE HERE: return logits of shape (B, 2)
        pass

# ── E3.4 — Token classification head ─────────────────────────────────────────
class TokenClassificationHead(nn.Module):
    def __init__(self, d_model, num_labels, dropout=0.1):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, sequence_output, labels=None):
        # logits: (B, S, num_labels)
        # If labels provided, compute cross-entropy ignoring -100 (padding)
        pass

# Quick test
clf = BERTForSequenceClassification(BERT_VOCAB, num_classes=2,
      d_model=D_MODEL, num_heads=NUM_HEADS, num_layers=N_LAYERS,
      d_ff=D_FF, max_len=SEQ_LEN).to(DEVICE)
out = clf(input_ids)
print(f"Classification output shape: {out.shape}")


## Section 4 — Flash Attention & Memory-Efficient Attention

### Why does vanilla attention use so much memory?

Vanilla scaled dot-product attention computes an `(S × S)` attention matrix explicitly, requiring **O(S²)** memory.  
For `S = 4096` and `B = 4` this is already hundreds of MB just for ​intermediates.

### Flash Attention (Dao et al., 2022)

Flash Attention fuses the attention kernel into a single CUDA kernel using **online softmax** and **tiling** so that intermediate matrices never leave SRAM.

| Property | Vanilla Attention | Flash Attention |
|---|---|---|
| Memory | O(S²) | O(S) |
| Speed | 1× | 2–4× faster |
| Numerical | Same | Identical |

### PyTorch 2.0+ built-in support

```python
# Automatically selects Flash-Attn, memory-efficient, or math backend
F.scaled_dot_product_attention(Q, K, V, is_causal=True)
```

Set the backend explicitly with `torch.backends.cuda.sdp_kernel(...)`.

### Memory comparison strategy

We will benchmark:
1. **Naive SDPA** — materialize full `(B, H, S, S)` score matrix
2. **PyTorch SDPA** — let PyTorch pick the best fused kernel
3. **Measure** peak memory with `torch.cuda.max_memory_allocated()`


In [ ]:
import torch.nn.functional as F
import math, gc, time

# ── Helpers ───────────────────────────────────────────────────────────────────
def reset_peak_memory():
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()

def peak_memory_mb():
    if DEVICE == "cuda":
        return torch.cuda.max_memory_allocated() / 1024**2
    return 0.0

# ── 1. Naive attention (materialise full score matrix) ────────────────────────
def naive_sdpa(Q, K, V, is_causal=True):
    """Vanilla O(S²) attention — useful baseline."""
    d_k  = Q.size(-1)
    S    = Q.size(-2)
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)   # (B, H, S, S)
    if is_causal:
        mask   = torch.triu(torch.ones(S, S, device=Q.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))
    attn = scores.softmax(-1)
    return attn @ V

# ── 2. Flash-Attn via PyTorch 2.0 ─────────────────────────────────────────────
def flash_sdpa(Q, K, V, is_causal=True):
    """Uses PyTorch SDPA dispatcher (Flash / mem-efficient / math fallback)."""
    return F.scaled_dot_product_attention(Q, K, V, is_causal=is_causal)

# ── Benchmark ─────────────────────────────────────────────────────────────────
def benchmark_attention(fn, Q, K, V, label, warmup=2, reps=10):
    for _ in range(warmup):
        fn(Q, K, V)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    reset_peak_memory()
    t0 = time.perf_counter()
    for _ in range(reps):
        out = fn(Q, K, V)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elapsed = (time.perf_counter() - t0) / reps * 1000
    mem = peak_memory_mb()
    print(f"  {label:<25} | time: {elapsed:6.2f} ms | peak mem: {mem:7.2f} MB | output: {out.shape}")

# Use a moderate sequence length so the difference is visible
B_bench, H_bench, S_bench, D_bench = 2, 8, 512, 64
dtype = torch.float16 if DEVICE == "cuda" else torch.float32

Q = torch.randn(B_bench, H_bench, S_bench, D_bench, device=DEVICE, dtype=dtype)
K = torch.randn_like(Q)
V = torch.randn_like(Q)

print("=== Attention Benchmark ===")
benchmark_attention(naive_sdpa, Q, K, V, "Naive SDPA")
benchmark_attention(flash_sdpa, Q, K, V, "PyTorch SDPA (fused)")

# ── Flash-Attn inside a Transformer block ────────────────────────────────────
class FlashMHA(nn.Module):
    """Multi-head attention backed by F.scaled_dot_product_attention."""
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.h  = num_heads
        self.dh = d_model // num_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj  = nn.Linear(d_model, d_model, bias=False)
        self.drop_p    = dropout

    def forward(self, x, is_causal=True):
        B, S, _ = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)                      # 3×(B,S,D)
        Q, K, V = [t.view(B, S, self.h, self.dh).transpose(1, 2) for t in qkv]
        # F.sdpa uses flash-attention when available
        attn_out = F.scaled_dot_product_attention(Q, K, V,
                       dropout_p=self.drop_p if self.training else 0.0,
                       is_causal=is_causal)
        out = attn_out.transpose(1, 2).contiguous().view(B, S, -1)
        return self.out_proj(out)

class FlashGPTBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = FlashMHA(d_model, num_heads, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ffn  = PositionwiseFFN(d_model, d_ff, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

# Quick integration test
flash_block = FlashGPTBlock(D_MODEL, NUM_HEADS, D_FF).to(DEVICE)
x_in = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=DEVICE)
x_out = flash_block(x_in)
print(f"\nFlashGPTBlock output: {x_out.shape}  ✓")


### 🏋️ Exercises — Section 4: Flash Attention

**E4.1** Modify `FlashMHA` to accept an explicit `padding_mask` tensor of shape `(B, S)` and pass it to `F.scaled_dot_product_attention` as an `attn_mask`.

**E4.2** Profile naive vs. fused SDPA at sequence lengths `[128, 256, 512, 1024, 2048]` and plot memory vs. sequence length using matplotlib.

**E4.3** Implement `sliding_window_sdpa`: break a long sequence into windows of size `W` with overlap `O`, apply `F.scaled_dot_product_attention` inside each window, and reassemble. Verify outputs match within a tolerance.

**E4.4** Build a `FlashGPT` model that uses `FlashGPTBlock` and benchmark training throughput (tokens/sec) against the standard `GPT` model from Section 2.


In [ ]:
# ── E4.1 — FlashMHA with padding mask ────────────────────────────────────────
class FlashMHAWithMask(FlashMHA):
    def forward(self, x, padding_mask=None, is_causal=True):
        B, S, _ = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        Q, K, V = [t.view(B, S, self.h, self.dh).transpose(1, 2) for t in qkv]

        attn_mask = None
        if padding_mask is not None:
            # YOUR CODE HERE:
            # padding_mask: (B, S) bool, True = padded
            # Convert to additive mask (B, 1, 1, S) with -inf at padded positions
            pass

        attn_out = F.scaled_dot_product_attention(Q, K, V, attn_mask=attn_mask,
                       dropout_p=self.drop_p if self.training else 0.0,
                       is_causal=is_causal if attn_mask is None else False)
        out = attn_out.transpose(1, 2).contiguous().view(B, S, -1)
        return self.out_proj(out)

# ── E4.2 — Memory vs sequence length plot ────────────────────────────────────
def memory_vs_seqlen(seq_lens, B=1, H=4, D=64, device=DEVICE):
    naive_mems, fused_mems = [], []
    for S in seq_lens:
        dtype = torch.float16 if device == "cuda" else torch.float32
        Q = torch.randn(B, H, S, D, device=device, dtype=dtype)
        K, V = torch.randn_like(Q), torch.randn_like(Q)

        # YOUR CODE HERE:
        # reset_peak_memory(); naive_sdpa(Q,K,V); naive_mems.append(peak_memory_mb())
        # reset_peak_memory(); flash_sdpa(Q,K,V); fused_mems.append(peak_memory_mb())
        naive_mems.append(0); fused_mems.append(0)

    return naive_mems, fused_mems

seq_lens = [128, 256, 512, 1024]
n_mem, f_mem = memory_vs_seqlen(seq_lens)
print("Seq  | Naive MB | Fused MB")
for s, n, f in zip(seq_lens, n_mem, f_mem):
    print(f"{s:<5}| {n:8.2f} | {f:8.2f}")

# ── E4.3 — Sliding-window SDPA (stub) ────────────────────────────────────────
def sliding_window_sdpa(Q, K, V, window_size=128, overlap=16):
    """YOUR CODE HERE: process in windows and reassemble."""
    # fallback: just use full SDPA
    return F.scaled_dot_product_attention(Q, K, V, is_causal=True)

# ── E4.4 — FlashGPT throughput benchmark (stub) ───────────────────────────────
# YOUR CODE HERE:
# 1. Build FlashGPT using FlashGPTBlock with the same hyperparameters as GPT.
# 2. Run N training steps with a random dataset and measure tokens/sec.
# 3. Compare against GPT from Section 2.
print("Stubs loaded — implement E4.1–E4.4 above.")


## Section 5 — `torch.compile`: Graph Capture & JIT Optimisation

### What is `torch.compile`?

Introduced in PyTorch 2.0, `torch.compile` uses **TorchDynamo** (Python bytecode capture) + **TorchInductor** (code generation for CPU/CUDA Triton) to JIT compile your model.

### Compilation modes

| Mode | Description | When to use |
|---|---|---|
| `"default"` | Balanced compile time / speedup | Dev → first try |
| `"reduce-overhead"` | Minimise CUDA kernel launch overhead | Small-batch inference |
| `"max-autotune"` | Intensive autotuning (Triton) | Long training runs |

### Usage pattern

```python
model = torch.compile(model, mode="reduce-overhead", fullgraph=False)
# First forward is slow (compilation); subsequent calls are fast
```

### Key concepts

- **`fullgraph=True`** — fail if the graph cannot be fully captured (no Python fallbacks). Useful to audit model for graph breaks.
- **Graph breaks** — caused by data-dependent control flow, `print()`, custom C++ ops, etc. Profile with `torch._dynamo.explain(model)(inputs)`.
- **`dynamic=True`** — handle variable sequence lengths without recompilation.


In [ ]:
import time

# ── Helper: measure latency ───────────────────────────────────────────────────
def measure_latency(model, inputs, warmup=3, reps=20, label=""):
    model.eval()
    with torch.no_grad():
        for _ in range(warmup):
            model(*inputs) if isinstance(inputs, (list, tuple)) else model(inputs)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(reps):
            model(*inputs) if isinstance(inputs, (list, tuple)) else model(inputs)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / reps * 1000
    print(f"  {label:<35} {ms:7.2f} ms/step")
    return ms

# ── Build a small GPT for benchmarking ────────────────────────────────────────
class GPTConfig:
    vocab_size = VOCAB_SIZE; seq_len = SEQ_LEN
    d_model = D_MODEL; num_heads = NUM_HEADS; n_layers = N_LAYERS
    d_ff = D_FF; dropout = 0.1

# Reuse GPT from Section 2
eager_model = GPT(GPTConfig.vocab_size, GPTConfig.d_model, GPTConfig.num_heads,
                  GPTConfig.n_layers, GPTConfig.d_ff, GPTConfig.seq_len,
                  GPTConfig.dropout).to(DEVICE)
eager_model.eval()

dummy_ids = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN), device=DEVICE)

print("=== torch.compile Benchmark ===")
t_eager = measure_latency(eager_model, (dummy_ids,), label="Eager (no compile)")

# Compile with different modes
for mode in ["default", "reduce-overhead"]:
    try:
        compiled = torch.compile(eager_model, mode=mode, dynamic=False)
        # Trigger compilation (first call is slower — not counted)
        with torch.no_grad():
            compiled(dummy_ids)
        t_c = measure_latency(compiled, (dummy_ids,), label=f"torch.compile mode={mode}")
        print(f"    Speedup vs eager: {t_eager/t_c:.2f}×")
    except Exception as e:
        print(f"  torch.compile({mode}) not available: {e}")

# ── Graph-break inspection ────────────────────────────────────────────────────
print("\n=== Graph-break analysis ===")
try:
    explanation = torch._dynamo.explain(eager_model)(dummy_ids)
    print(f"  Number of graphs captured : {explanation.graph_count}")
    print(f"  Number of graph breaks    : {explanation.break_reasons.__len__() if hasattr(explanation, 'break_reasons') else 'n/a'}")
except Exception as e:
    print(f"  Could not run dynamo explain: {e}")

# ── compile() + AMP combined ──────────────────────────────────────────────────
print("\n=== compile + AMP ===")
try:
    compiled_amp = torch.compile(eager_model, mode="default")
    with torch.no_grad(), torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=DEVICE=="cuda"):
        out = compiled_amp(dummy_ids)
    print(f"  compile + AMP output shape: {out.shape}  ✓")
except Exception as e:
    print(f"  compile + AMP skipped: {e}")


### 🏋️ Exercises — Section 5: torch.compile

**E5.1** Try `fullgraph=True` on the `GPT` model. What graph breaks are reported? Fix at least one by refactoring the offending code.

**E5.2** Compile only the attention sub-module (`model.blocks[0].attn`) instead of the entire model. Measure whether selective compilation gives a better warm-up latency than whole-model compilation.

**E5.3** Use `dynamic=True` to compile the model once and time it on three different sequence lengths `[32, 64, 128]`. Compare recompilation count vs. `dynamic=False`.

**E5.4** Profile the compiled model with `torch.profiler.profile` and compare the CUDA kernel timeline against the eager model. Which kernels are fused in the compiled version?


In [ ]:
# ── E5.1 — fullgraph=True ─────────────────────────────────────────────────────
try:
    compiled_full = torch.compile(eager_model, fullgraph=True)
    with torch.no_grad():
        compiled_full(dummy_ids)
    print("E5.1: fullgraph=True compiled successfully!")
except Exception as e:
    print(f"E5.1 graph break detected: {e}")
    # YOUR CODE HERE: identify and fix the graph break

# ── E5.2 — Selective module compilation ───────────────────────────────────────
# YOUR CODE HERE:
# compiled_attn = torch.compile(eager_model.blocks[0].attn, mode="default")
# Replace the first block's attention module with the compiled version
# then benchmark

# ── E5.3 — Dynamic shapes ─────────────────────────────────────────────────────
try:
    compiled_dynamic = torch.compile(eager_model, dynamic=True)
    for seq in [32, 64, 128]:
        inp = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, seq), device=DEVICE)
        with torch.no_grad():
            out = compiled_dynamic(inp)
        print(f"E5.3: seq_len={seq} → output {out.shape}")
except Exception as e:
    print(f"E5.3: {e}")

# ── E5.4 — Profiler comparison stub ───────────────────────────────────────────
# YOUR CODE HERE:
# from torch.profiler import profile, ProfilerActivity
# with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
#     with torch.no_grad():
#         compiled(dummy_ids)
# print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

print("Stubs loaded — implement E5.1–E5.4 above.")


## Section 6 — Distributed Data Parallel (DDP) Training

### Why distributed training?

Single-GPU memory caps out at ~80 GB (A100). Long-context LLMs or large batch sizes require spreading work across multiple GPUs or nodes.

### DDP vs. DataParallel

| | `DataParallel` (legacy) | `DistributedDataParallel` |
|---|---|---|
| Process model | 1 process, N threads | N processes (1 per GPU) |
| GIL bottleneck | Yes | No |
| All-reduce | Asymmetric (GPU 0 aggregates) | Symmetric (NCCL ring) |
| Recommended | ❌ | ✅ |

### DDP lifecycle

```
dist.init_process_group("nccl")
model = model.to(rank)
model = DDP(model, device_ids=[rank])
# ... training loop ...
dist.destroy_process_group()
```

### Launch pattern (torchrun)

```bash
torchrun --nproc_per_node=4 train.py
```

Inside `train.py`, rank is available as `int(os.environ["LOCAL_RANK"])`.

### Gradient synchronisation

DDP overlaps **gradient all-reduce** with the backward pass automatically (bucket-based). Use `model.no_sync()` context manager for gradient accumulation.


In [ ]:
"""
DDP cannot be demonstrated inline (it requires launching multiple processes).
This cell provides:
  1. A self-contained `ddp_train_script.py` that can be run with torchrun.
  2. A single-GPU simulation using fake process-group init with gloo backend.
"""

import os, textwrap

# ── Write a ready-to-run DDP training script ─────────────────────────────────
DDP_SCRIPT = textwrap.dedent("""
import os, torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler, TensorDataset

# ── Hyper-parameters ──────────────────────────────────────────────────────────
VOCAB_SIZE = 256; SEQ_LEN = 64; BATCH = 8; D_MODEL = 128
NUM_HEADS = 8; N_LAYERS = 4; D_FF = 512; STEPS = 50

# ── Init process group ────────────────────────────────────────────────────────
def setup(rank, world_size):
    os.environ.setdefault("MASTER_ADDR", "localhost")
    os.environ.setdefault("MASTER_PORT", "12355")
    dist.init_process_group("nccl" if torch.cuda.is_available() else "gloo",
                            rank=rank, world_size=world_size)
    if torch.cuda.is_available():
        torch.cuda.set_device(rank)

def cleanup():
    dist.destroy_process_group()

# ── Toy dataset ───────────────────────────────────────────────────────────────
def get_dataloader(rank, world_size, n=512):
    ids  = torch.randint(0, VOCAB_SIZE, (n, SEQ_LEN))
    ds   = TensorDataset(ids)
    samp = DistributedSampler(ds, num_replicas=world_size, rank=rank, shuffle=True)
    return DataLoader(ds, batch_size=BATCH, sampler=samp)

# ── Minimal GPT (must be importable here; paste or import from shared utils) ──
import torch.nn as nn
class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.lm_head = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False)
    def forward(self, x):
        return self.lm_head(self.embed(x))

# ── Training function (one process) ──────────────────────────────────────────
def train(rank, world_size):
    setup(rank, world_size)
    device = torch.device(f"cuda:{rank}" if torch.cuda.is_available() else "cpu")
    model  = TinyGPT().to(device)
    model  = DDP(model, device_ids=[rank] if torch.cuda.is_available() else None)
    opt    = torch.optim.AdamW(model.parameters(), lr=3e-4)
    loader = get_dataloader(rank, world_size)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for step, (ids,) in enumerate(loader):
        if step >= STEPS: break
        ids = ids.to(device)
        logits = model(ids[:, :-1])            # (B, S-1, V)
        loss   = criterion(logits.reshape(-1, VOCAB_SIZE), ids[:, 1:].reshape(-1))
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if rank == 0 and step % 10 == 0:
            print(f"  step {step:3d} | loss {loss.item():.4f}")

    # Save only from rank 0
    if rank == 0:
        torch.save(model.module.state_dict(), "ddp_ckpt.pt")
        print("Checkpoint saved: ddp_ckpt.pt")
    cleanup()

if __name__ == "__main__":
    world_size = torch.cuda.device_count() or 1
    if world_size > 1:
        import torch.multiprocessing as mp
        mp.spawn(train, args=(world_size,), nprocs=world_size, join=True)
    else:
        # Fallback: single process with fake DDP (gloo)
        os.environ["MASTER_ADDR"] = "localhost"
        os.environ["MASTER_PORT"] = "12356"
        train(rank=0, world_size=1)
""")

script_path = "/teamspace/studios/this_studio/ddp_train_script.py"
with open(script_path, "w") as f:
    f.write(DDP_SCRIPT.strip())
print(f"DDP script written to {script_path}")
print("Run it with:\n  torchrun --nproc_per_node=NUM_GPUS ddp_train_script.py")

# ── Key DDP patterns (single-node annotation) ─────────────────────────────────
print("""
Key DDP API summary
────────────────────────────────────────────────────────────
dist.init_process_group(backend)  — 'nccl' GPU / 'gloo' CPU
DDP(model, device_ids=[rank])     — wrap after .to(device)
DistributedSampler                — ensures non-overlapping data per GPU
model.module                      — unwrap DDP to access the raw model
model.no_sync()                   — suppress all-reduce for gradient accumulation
dist.barrier()                    — synchronise all ranks
dist.destroy_process_group()      — clean up when done
────────────────────────────────────────────────────────────""")


### 🏋️ Exercises — Section 6: Distributed Training (DDP)

**E6.1** Modify `ddp_train_script.py` to use gradient accumulation with `model.no_sync()` for 4 micro-steps before calling `optimizer.step()`. Verify that the effective batch size is `BATCH × world_size × 4`.

**E6.2** Add `torch.cuda.amp.GradScaler` to the DDP training loop. Make sure the scaler state is not split across ranks (hint: only rank 0 needs to save/load it).

**E6.3** Replace the `DistributedSampler` with a custom shuffle strategy that ensures the same tokens never appear in two ranks during the same epoch.

**E6.4** (Research) Explain the difference between **DDP** (data parallelism), **FSDP** (fully sharded data parallelism), and **tensor parallelism**. Sketch a diagram of how gradients flow in each case.


## Section 7 — Memory Profiling & Leak Detection

### Memory budget breakdown (LLM training)

```
Total GPU memory = Parameters + Gradients + Optimiser states + Activations + Overhead
   AdamW (FP32) : 4× param bytes  (params, grad, m, v)
   Mixed prec   : parameters × 6 bytes  (FP16 params + FP32 master copy + grad)
```

### Tools

| Tool | What it measures |
|---|---|
| `torch.cuda.memory_summary()` | Snapshot: allocated, reserved, fragmentation |
| `torch.cuda.memory_stats()` | Fine-grained stats dict |
| `torch.profiler.profile` | Timeline: allocations per op |
| `tracemalloc` (CPU) | Python-level heap allocations |

### Common memory leaks

1. **Accumulating tensors in a list** — `losses.append(loss)` instead of `.item()`
2. **Not calling `optimizer.zero_grad()`** — gradient graph accumulates
3. **Holding references to intermediate activations** — storing `output` instead of `output.detach()`
4. **Persistent cache tensors** — keys/values not freed between decode steps


In [ ]:
import gc
from torch.profiler import profile, ProfilerActivity, tensorboard_trace_handler, schedule

# ── 1. Quick memory snapshot helper ─────────────────────────────────────────
def memory_snapshot(label=""):
    if DEVICE != "cuda":
        print(f"[{label}] CUDA not available — skipping GPU memory snapshot")
        return
    stats = torch.cuda.memory_stats()
    alloc = stats["allocated_bytes.all.current"] / 1024**2
    reserv= stats["reserved_bytes.all.current"]  / 1024**2
    peak  = stats["allocated_bytes.all.peak"]     / 1024**2
    print(f"[{label}] allocated={alloc:.1f}MB  reserved={reserv:.1f}MB  peak={peak:.1f}MB")

# ── 2. Detect leak: accumulating loss tensors ────────────────────────────────
print("=== Demo: memory leak vs. correct code ===")
model = eager_model.train()
opt   = torch.optim.AdamW(model.parameters(), lr=1e-4)

# -- Leaky version (stores full tensor with computation graph) --
leaky_losses = []
torch.cuda.reset_peak_memory_stats() if DEVICE == "cuda" else None
for _ in range(5):
    ids    = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN), device=DEVICE)
    logits = model(ids[:, :-1])
    loss   = F.cross_entropy(logits.view(-1, VOCAB_SIZE), ids[:, 1:].reshape(-1))
    leaky_losses.append(loss)               # ← BUG: retains computation graph!
    opt.zero_grad(); loss.backward(); opt.step()
memory_snapshot("After leaky loop")

# Fix: use .item() to detach scalar
correct_losses = []
torch.cuda.reset_peak_memory_stats() if DEVICE == "cuda" else None
for _ in range(5):
    ids    = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN), device=DEVICE)
    logits = model(ids[:, :-1])
    loss   = F.cross_entropy(logits.view(-1, VOCAB_SIZE), ids[:, 1:].reshape(-1))
    correct_losses.append(loss.item())      # ← correct: detaches from graph
    opt.zero_grad(); loss.backward(); opt.step()
memory_snapshot("After correct loop")

# ── 3. torch.profiler: operation-level memory timeline ──────────────────────
print("\n=== torch.profiler memory trace ===")
activities = [ProfilerActivity.CPU]
if DEVICE == "cuda":
    activities.append(ProfilerActivity.CUDA)

model.eval()
with profile(activities=activities,
             profile_memory=True,
             record_shapes=True,
             with_stack=False) as prof:
    with torch.no_grad():
        for _ in range(3):
            model(dummy_ids)

print(prof.key_averages().table(sort_by="self_cpu_memory_usage", row_limit=10))

# ── 4. How to find actual peak memory per operator ──────────────────────────
print("\n=== Peak memory per operator ===")
try:
    mem_table = prof.key_averages().table(sort_by="self_cuda_memory_usage", row_limit=5)
    print(mem_table)
except Exception:
    print("(CUDA memory table not available — run on GPU to see per-op GPU memory)")

# ── 5. cuda.memory_summary ──────────────────────────────────────────────────
if DEVICE == "cuda":
    print("\n=== cuda.memory_summary (excerpt) ===")
    summary = torch.cuda.memory_summary(abbreviated=True)
    print(summary[:1500])
else:
    print("\n(cuda.memory_summary available on CUDA devices only)")


### 🏋️ Exercises — Section 7: Memory Profiling

**E7.1** Write a `memory_audit(model, input_ids)` function that reports:
- Parameter memory (MB)
- Gradient memory after one backward pass (MB)
- Activation memory during forward (estimated via peak delta)

**E7.2** Introduce 3 different memory leaks into the training loop (e.g., store tensors, forget `zero_grad`, hold a reference to `loss` from a previous iteration). Use `torch.profiler` to detect each leak without looking at the code.

**E7.3** Use `torch.cuda.memory_snapshot()` (PyTorch ≥ 2.1) to capture a full allocation timeline and visualise it with the `torch.cuda._memory_viz` module (or the Memory Viz browser tool at https://pytorch.org/memory_viz).

**E7.4** Profile the GPT model forward pass and identify which operation consumes the most memory. Reduce it by applying gradient checkpointing (preview of Section 9) and measure the reduction.


In [ ]:
# ── E7.1 — Memory audit ───────────────────────────────────────────────────────
def memory_audit(model, input_ids):
    """Report parameter, gradient, and estimated activation memory."""
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    print(f"  Parameter memory : {param_bytes/1024**2:.2f} MB")

    # YOUR CODE HERE:
    # 1. Reset peak memory, run forward+backward, measure delta for gradients
    # 2. Estimate activation memory = peak_after_forward - baseline - param_bytes
    print("  Gradient / activation memory: implement me!")

memory_audit(eager_model.train(), dummy_ids)

# ── E7.2 — Introduce and detect leaks (stubs) ────────────────────────────────
# YOUR CODE HERE:
# Introduce each of the 3 leaks below and profile each variant with torch.profiler
# Leak A: store loss tensor without .item()
# Leak B: forget optimizer.zero_grad() for several steps
# Leak C: accumulate activations by storing model output tensors

# ── E7.3 — Memory snapshot + viz (PyTorch ≥ 2.1 only) ───────────────────────
try:
    torch.cuda.memory._record_memory_history(max_entries=100_000)
    with torch.no_grad():
        _ = eager_model(dummy_ids)
    snapshot = torch.cuda.memory._snapshot()
    torch.cuda.memory._record_memory_history(enabled=None)
    # Save for memory_viz tool
    import pickle
    with open("memory_snapshot.pkl", "wb") as f:
        pickle.dump(snapshot, f)
    print("E7.3: snapshot saved to memory_snapshot.pkl")
    print("  Visualise at: https://pytorch.org/memory_viz (upload the .pkl file)")
except Exception as e:
    print(f"E7.3: memory snapshot not available ({e})")

print("Stubs loaded — implement E7.1–E7.4 above.")


## Section 8 — Training Monitoring with TensorBoard

### What to log

| Signal | Why it matters |
|---|---|
| Loss (train / val) | Model is learning; check for divergence |
| Gradient norms | Detect vanishing / exploding gradients |
| Parameter histograms | Identify dead neurons or layer collapse |
| Learning rate | Confirm scheduler behaviour |
| Attention weight heatmaps | Inspect what the model attends to |
| GPU utilisation | Identify bottlenecks |

### TensorBoard API quick-reference

```python
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter(log_dir="runs/my_experiment")

writer.add_scalar("loss/train", loss, step)
writer.add_histogram("grads/block0", model.blocks[0].attn.out_proj.weight.grad, step)
writer.add_image("attn_map", attn_img, step, dataformats="HW")
writer.add_hparams({"lr": 1e-3, "batch": 8}, {"hparam/best_loss": best_loss})
writer.close()
```

Launch: `tensorboard --logdir runs/ --port 6006`


In [ ]:
try:
    from torch.utils.tensorboard import SummaryWriter
    HAS_TB = True
except ImportError:
    HAS_TB = False
    print("tensorboard not installed — run: pip install tensorboard")

import shutil, pathlib

# ── Clean stale run ───────────────────────────────────────────────────────────
LOG_DIR = "/teamspace/studios/this_studio/runs/advanced_s8"
if pathlib.Path(LOG_DIR).exists():
    shutil.rmtree(LOG_DIR)

# ── GPT with attention-map hooks ─────────────────────────────────────────────
class MonitoredMHA(MultiHeadAttention):
    """Stores last attention weights for logging."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.last_attn_weights = None

    def forward(self, q, k, v, mask=None):
        B, S, _ = q.shape
        def _proj(x, w): return x @ w.t()
        Q = _proj(q, self.w_q).view(B, S, self.h, self.dh).transpose(1,2)
        K = _proj(k, self.w_k).view(B, S, self.h, self.dh).transpose(1,2)
        V = _proj(v, self.w_v).view(B, S, self.h, self.dh).transpose(1,2)
        scores = (Q @ K.transpose(-2,-1)) / (self.dh ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), float("-inf"))
        weights = scores.softmax(-1)
        self.last_attn_weights = weights.detach()       # save for logging
        out = (weights @ V).transpose(1,2).contiguous().view(B, S, -1)
        return self.out(out)

# ── Mini training loop with TensorBoard logging ──────────────────────────────
def monitored_training(steps=30, log_every=5):
    model = GPT(VOCAB_SIZE, D_MODEL, NUM_HEADS, N_LAYERS, D_FF, SEQ_LEN).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=3e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)

    writer = SummaryWriter(LOG_DIR) if HAS_TB else None
    model.train()

    for step in range(steps):
        ids = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN), device=DEVICE)
        logits = model(ids[:, :-1])
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), ids[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0).item()
        opt.step(); sched.step()

        if writer and step % log_every == 0:
            writer.add_scalar("loss/train", loss.item(), step)
            writer.add_scalar("grad_norm", grad_norm, step)
            writer.add_scalar("lr", sched.get_last_lr()[0], step)
            # Log parameter histograms every 10 steps
            if step % 10 == 0:
                for name, param in model.named_parameters():
                    writer.add_histogram(f"params/{name}", param.data, step)
                    if param.grad is not None:
                        writer.add_histogram(f"grads/{name}", param.grad, step)

        if step % log_every == 0:
            print(f"  step {step:3d} | loss {loss.item():.4f} | grad_norm {grad_norm:.3f} | lr {sched.get_last_lr()[0]:.2e}")

    if writer:
        writer.close()
        print(f"\nTensorBoard logs written to {LOG_DIR}")
        print("Launch with: tensorboard --logdir runs/ --port 6006")

monitored_training()


### 🏋️ Exercises — Section 8: Training Monitoring

**E8.1** Add a validation loss curve to `monitored_training` by evaluating on a held-out random batch every `log_every` steps. Plot both train and val loss on the same TensorBoard scalar panel.

**E8.2** Log the attention weight heatmap for head 0 of layer 0 every 10 steps. Use `writer.add_image` with a normalised `(1, S, S)` tensor. Observe how attention patterns evolve.

**E8.3** Use `writer.add_hparams` to compare two runs: one with `lr=1e-3` and one with `lr=1e-4`. Which converges faster? Which achieves a lower final loss?

**E8.4** Implement a `GradientMonitor` callback class with a `on_after_backward(model, step, writer)` method that logs per-layer gradient norms as a histogram and raises a warning if any layer's norm exceeds a threshold.


## Section 9 — Gradient Checkpointing

### The memory/compute tradeoff

During backpropagation PyTorch caches all intermediate activations from the forward pass. For a model with L layers and activation size A per layer, peak memory is O(L·A).

**Gradient checkpointing** discards activations after the forward pass and **recomputes** them during the backward pass.

```
Memory  : O(√L · A)   (with optimal segmentation)
Compute : +33% extra forward passes (approx.)
```

### API

```python
from torch.utils.checkpoint import checkpoint

def forward(self, x):
    for block in self.blocks:
        x = checkpoint(block, x, use_reentrant=False)
    return x
```

### When to use

- Fine-tuning large models that barely fit in VRAM
- Increasing effective batch size without more GPUs
- Long-context models where activation memory is the bottleneck

> **Note:** `use_reentrant=False` is the modern recommended mode — it plays well with `torch.compile` and DDP.


In [ ]:
from torch.utils.checkpoint import checkpoint as ckpt_fn

# ── GPT with gradient checkpointing ──────────────────────────────────────────
class GPTWithCheckpointing(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, n_layers, d_ff, max_len,
                 dropout=0.1, use_checkpointing=True):
        super().__init__()
        self.use_ckpt = use_checkpointing
        self.tok_emb  = nn.Embedding(vocab_size, d_model)
        self.pos_emb  = nn.Embedding(max_len, d_model)
        self.drop     = nn.Dropout(dropout)
        self.blocks   = nn.ModuleList([GPTBlock(d_model, num_heads, d_ff, dropout)
                                       for _ in range(n_layers)])
        self.ln_f     = nn.LayerNorm(d_model)
        self.lm_head  = nn.Linear(d_model, vocab_size, bias=False)
        self.tok_emb.weight = self.lm_head.weight   # weight tying

    def forward(self, idx):
        B, S = idx.shape
        pos  = torch.arange(S, device=idx.device).unsqueeze(0)
        x    = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            if self.use_ckpt and self.training:
                x = ckpt_fn(block, x, use_reentrant=False)
            else:
                x = block(x)
        return self.lm_head(self.ln_f(x))

# ── Benchmark: memory with vs without checkpointing ──────────────────────────
def benchmark_checkpointing(n_layers=8, seq_len=128, batch_size=4):
    results = {}
    for use_ckpt in [False, True]:
        gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

        model = GPTWithCheckpointing(VOCAB_SIZE, D_MODEL, NUM_HEADS, n_layers,
                                     D_FF, seq_len, use_checkpointing=use_ckpt).to(DEVICE)
        model.train()
        opt = torch.optim.AdamW(model.parameters(), lr=1e-4)

        ids = torch.randint(0, VOCAB_SIZE, (batch_size, seq_len), device=DEVICE)
        logits = model(ids[:, :-1])
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), ids[:, 1:].reshape(-1))
        loss.backward()
        opt.step()

        mem_mb = peak_memory_mb()
        label  = "checkpointing=ON " if use_ckpt else "checkpointing=OFF"
        results[label] = mem_mb
        print(f"  {label} | peak memory: {mem_mb:.1f} MB")

    if all(v > 0 for v in results.values()):
        ratio = list(results.values())[0] / list(results.values())[1]
        print(f"  Memory saving: {ratio:.2f}×  (baseline / checkpointed)")

print("=== Gradient Checkpointing Benchmark ===")
benchmark_checkpointing()

print("\nNote: with 8 layers the saving is modest; scales to 2–3× with 24+ layers.")


### 🏋️ Exercises — Section 9: Gradient Checkpointing

**E9.1** Benchmark `GPTWithCheckpointing` at N_LAYERS ∈ {4, 8, 16, 32}. Plot peak memory vs. number of layers with and without checkpointing.

**E9.2** Wrap only the **first half** of transformer blocks with `checkpoint` and leave the second half eager. Is this better than wrapping all blocks? Why might selective checkpointing be useful?

**E9.3** Implement **activation offloading**: after each block's forward, move activations to CPU RAM and move them back before the backward pass. Measure the memory vs. latency tradeoff vs. standard checkpointing.

**E9.4** Verify gradient correctness: run a full forward+backward with and without checkpointing on the same random seed, and assert that gradients match within `atol=1e-4`.


In [ ]:
# ── E9.1 — Memory vs n_layers ─────────────────────────────────────────────────
for n in [4, 8]:   # extend to [16, 32] on large GPU
    print(f"\n--- n_layers={n} ---")
    benchmark_checkpointing(n_layers=n)

# ── E9.2 — Selective checkpointing ────────────────────────────────────────────
class GPTSelectiveCheckpoint(GPTWithCheckpointing):
    def forward(self, idx):
        B, S = idx.shape
        pos  = torch.arange(S, device=idx.device).unsqueeze(0)
        x    = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        mid  = len(self.blocks) // 2
        for i, block in enumerate(self.blocks):
            # YOUR CODE HERE: ckpt_fn only for i < mid
            x = block(x)
        return self.lm_head(self.ln_f(x))

# ── E9.4 — Gradient correctness check ────────────────────────────────────────
def verify_checkpointing_gradients(seed=42):
    def run(use_ckpt):
        torch.manual_seed(seed)
        m = GPTWithCheckpointing(VOCAB_SIZE, D_MODEL, NUM_HEADS, 2, D_FF, SEQ_LEN,
                                 use_checkpointing=use_ckpt).to(DEVICE).train()
        ids = torch.randint(0, VOCAB_SIZE, (2, SEQ_LEN), device=DEVICE)
        logits = m(ids[:, :-1])
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), ids[:, 1:].reshape(-1))
        loss.backward()
        return {n: p.grad.clone() for n, p in m.named_parameters() if p.grad is not None}

    grads_eager = run(False)
    grads_ckpt  = run(True)
    for k in grads_eager:
        match = torch.allclose(grads_eager[k], grads_ckpt[k], atol=1e-4)
        if not match:
            print(f"  MISMATCH: {k}")
    print("E9.4: gradient check complete (any mismatches printed above)")

verify_checkpointing_gradients()


## Section 10 — KV-Cache & Fast Autoregressive Inference

### Why is autoregressive decoding slow?

Without a cache, generating token `t` requires recomputing keys and values for all previous `t-1` tokens. This means:
- **O(T²)** total FLOPs to generate a sequence of length T
- **O(T)** memory per step (just the new token's embedding)

### KV-Cache

Cache the K and V projections for all past tokens. At step t:
1. Only compute Q for the **new token** position.
2. Concatenate with cached K, V.
3. Run attention using full `(1, T)` key/value sequence.

```
Compute reduction: O(T²) → O(T)  per sequence  (linear in T)
Memory overhead  : O(2 · H · D · T) per layer per batch element
```

### Implementation

Each attention block returns `(output, (K_cache, V_cache))`.  
The model accumulates a `past_key_values` list across layers.  

### Static KV-cache (batched generation)

Pre-allocate a fixed-size buffer to avoid dynamic concatenation overhead.  
Used in production systems (vLLM, HuggingFace generate with `use_cache=True`).


In [ ]:
import math

# ── KV-Cache aware attention ──────────────────────────────────────────────────
class CachedMHA(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.h  = num_heads
        self.dh = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, past_kv=None):
        """
        x        : (B, S_new, D)   — S_new=1 during generation
        past_kv  : (K_past, V_past) tensors of shape (B, H, S_past, Dh), or None
        Returns  : output (B, S_new, D), new_kv (K, V)
        """
        B, S_new, _ = x.shape
        Q = self.w_q(x).view(B, S_new, self.h, self.dh).transpose(1, 2)
        K = self.w_k(x).view(B, S_new, self.h, self.dh).transpose(1, 2)
        V = self.w_v(x).view(B, S_new, self.h, self.dh).transpose(1, 2)

        if past_kv is not None:
            K_past, V_past = past_kv
            K = torch.cat([K_past, K], dim=2)   # (B, H, S_past+S_new, Dh)
            V = torch.cat([V_past, V], dim=2)

        attn = F.scaled_dot_product_attention(Q, K, V,
                   is_causal=(past_kv is None))  # causal only on full prefill

        out = attn.transpose(1, 2).contiguous().view(B, S_new, -1)
        return self.out(out), (K, V)

class CachedGPTBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = CachedMHA(d_model, num_heads)
        self.ln2  = nn.LayerNorm(d_model)
        self.ffn  = PositionwiseFFN(d_model, d_ff, dropout)

    def forward(self, x, past_kv=None):
        attn_out, new_kv = self.attn(self.ln1(x), past_kv)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x, new_kv

class CachedGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, n_layers, d_ff, max_len, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop    = nn.Dropout(dropout)
        self.blocks  = nn.ModuleList([CachedGPTBlock(d_model, num_heads, d_ff, dropout)
                                      for _ in range(n_layers)])
        self.ln_f    = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.tok_emb.weight = self.lm_head.weight

    def forward(self, idx, past_kvs=None, offset=0):
        B, S = idx.shape
        pos  = torch.arange(offset, offset + S, device=idx.device).unsqueeze(0)
        x    = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        new_kvs = []
        for i, block in enumerate(self.blocks):
            pkv = past_kvs[i] if past_kvs is not None else None
            x, nkv = block(x, pkv)
            new_kvs.append(nkv)
        return self.lm_head(self.ln_f(x)), new_kvs

    @torch.no_grad()
    def generate_with_cache(self, prompt_ids, max_new_tokens=50):
        """Autoregressive decoding with KV-Cache."""
        self.eval()
        B = prompt_ids.shape[0]
        ids = prompt_ids.clone()

        # Prefill: process the entire prompt at once
        _, past_kvs = self.forward(ids, past_kvs=None, offset=0)
        offset = ids.shape[1]

        for _ in range(max_new_tokens):
            last = ids[:, -1:]
            logits, past_kvs = self.forward(last, past_kvs=past_kvs, offset=offset)
            next_id = logits[:, -1, :].argmax(-1, keepdim=True)
            ids = torch.cat([ids, next_id], dim=1)
            offset += 1

        return ids

# ── Benchmark: with vs without KV-cache ──────────────────────────────────────
def benchmark_generation(prompt_len=16, gen_len=64):
    prompt = torch.randint(0, VOCAB_SIZE, (1, prompt_len), device=DEVICE)

    cached_model = CachedGPT(VOCAB_SIZE, D_MODEL, NUM_HEADS, N_LAYERS, D_FF,
                              prompt_len + gen_len + 10).to(DEVICE)
    plain_model  = GPT(VOCAB_SIZE, D_MODEL, NUM_HEADS, N_LAYERS, D_FF,
                       prompt_len + gen_len + 10).to(DEVICE)

    # Copy weights so comparison is fair
    with torch.no_grad():
        for p1, p2 in zip(plain_model.parameters(), cached_model.parameters()):
            p2.copy_(p1)

    print("=== Generation Benchmark ===")
    t0 = time.perf_counter()
    plain_model.generate(prompt, max_new_tokens=gen_len, temperature=1.0)
    t_plain = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    cached_model.generate_with_cache(prompt, max_new_tokens=gen_len)
    t_cached = (time.perf_counter() - t0) * 1000

    print(f"  Without KV-cache: {t_plain:.1f} ms  ({gen_len / t_plain * 1000:.0f} tok/s)")
    print(f"  With    KV-cache: {t_cached:.1f} ms  ({gen_len / t_cached * 1000:.0f} tok/s)")
    if t_plain > 0:
        print(f"  Speedup: {t_plain/t_cached:.2f}×")

benchmark_generation()


### 🏋️ Exercises — Section 10: KV-Cache

**E10.1** Add **beam search** (beam width = 4) on top of `CachedGPT.generate_with_cache`. Verify that beam search produces higher log-probability sequences than greedy.

**E10.2** Implement a **static KV-cache**: pre-allocate a `(B, H, MAX_SEQ, Dh)` tensor and use an integer pointer to write new K/V in-place, avoiding concatenation. Benchmark the memory allocation overhead vs. dynamic concatenation.

**E10.3** Extend `CachedGPT` to support **batch generation**: generate for B=8 sequences simultaneously, with different prompt lengths and proper padding mask handling.

**E10.4** Measure how KV-cache memory scales with sequence length: log `cache_size_mb` vs. `gen_len` for a 12-layer model and compare against the formula `2 × n_layers × H × Dh × T × element_size`.


## Section 11 — Model Quantization

### What is quantization?

Replace 32-bit (FP32) or 16-bit (FP16) weights/activations with lower-precision representations:

| Format | Bits | Size ratio | Use case |
|---|---|---|---|
| FP32 | 32 | 1× (baseline) | Training |
| BF16 / FP16 | 16 | 0.5× | AMP training / inference |
| INT8 | 8 | 0.25× | Inference speedup |
| INT4 / NF4 | 4 | 0.125× | QLoRA fine-tuning |

### Types of quantization

1. **Post-Training Quantization (PTQ)** — quantize after training, no data needed (fast, small accuracy loss)
2. **Quantization-Aware Training (QAT)** — fake-quantize during training (best accuracy)
3. **Dynamic quantization** — weights quantized, activations quantized at runtime

### PyTorch dynamic quantization (INT8)

```python
import torch.quantization
q_model = torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
```

Supported modules: `nn.Linear`, `nn.LSTM`, `nn.GRU`.


In [ ]:
import torch.quantization

# ── Helper: model size in MB ──────────────────────────────────────────────────
def model_size_mb(model):
    total = sum(p.nelement() * p.element_size() for p in model.parameters())
    total += sum(b.nelement() * b.element_size() for b in model.buffers())
    return total / 1024**2

# ── FP32 baseline model ───────────────────────────────────────────────────────
fp32_model = GPT(VOCAB_SIZE, D_MODEL, NUM_HEADS, N_LAYERS, D_FF, SEQ_LEN).cpu()
fp32_model.eval()
print(f"FP32 model size : {model_size_mb(fp32_model):.2f} MB")

# ── INT8 Dynamic Quantization ────────────────────────────────────────────────
int8_model = torch.quantization.quantize_dynamic(
    fp32_model, {nn.Linear}, dtype=torch.qint8
)
print(f"INT8 model size : {model_size_mb(int8_model):.2f} MB  "
      f"(+buffers — actual savings depend on quantized weight format)")

# ── FP16 (half precision) ─────────────────────────────────────────────────────
fp16_model = fp32_model.half()    # cast all parameters to FP16
print(f"FP16 model size : {model_size_mb(fp16_model):.2f} MB")

# ── Latency comparison (CPU) ──────────────────────────────────────────────────
dummy_cpu = dummy_ids.cpu()

def bench_model_cpu(model, inputs, reps=20, label=""):
    model.eval()
    with torch.no_grad():
        for _ in range(3): model(inputs)   # warmup
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(reps): model(inputs)
    ms = (time.perf_counter() - t0) / reps * 1000
    print(f"  {label:<20} {ms:7.2f} ms/forward")

print("\n=== Quantization Latency (CPU) ===")
bench_model_cpu(fp32_model,          dummy_cpu,          label="FP32 eager")
bench_model_cpu(int8_model,          dummy_cpu,          label="INT8 dynamic")
bench_model_cpu(fp16_model.float(),  dummy_cpu,          label="FP16→FP32 cast")

# ── Output accuracy check ─────────────────────────────────────────────────────
with torch.no_grad():
    out_fp32 = fp32_model(dummy_cpu).float()
    out_int8 = int8_model(dummy_cpu).float()
    out_fp16 = fp16_model(dummy_cpu.to(torch.float16)).float()

print(f"\nMax |FP32 - INT8| logit diff : {(out_fp32 - out_int8).abs().max():.4f}")
print(f"Max |FP32 - FP16| logit diff : {(out_fp32 - out_fp16).abs().max():.4f}")

# ── Perplexity degradation ────────────────────────────────────────────────────
def estimate_perplexity(model, ids, dtype=torch.float32):
    with torch.no_grad():
        inp    = ids[:, :-1].to(dtype)
        target = ids[:, 1:]
        logits = model(inp.long() if dtype == torch.float32 else inp.long())
        # Handle fp16 model: pass long ids
        logits = logits.float()
        loss   = F.cross_entropy(logits.view(-1, VOCAB_SIZE), target.reshape(-1))
    return torch.exp(loss).item()

ids_sample = torch.randint(0, VOCAB_SIZE, (4, SEQ_LEN)).cpu()
print(f"\nFP32 perplexity  : {estimate_perplexity(fp32_model, ids_sample):.2f}")
print(f"INT8 perplexity  : {estimate_perplexity(int8_model, ids_sample):.2f}")


### 🏋️ Exercises — Section 11: Quantization

**E11.1** Apply `quantize_dynamic` selectively to only the MLP layers (`PositionwiseFFN`) while leaving attention matrices in FP32. Measure size and perplexity vs. quantizing all linear layers.

**E11.2** Implement a simple **absmax INT8 quantizer** from scratch:
```
scale = max(|W|) / 127
W_int8 = round(W / scale).clamp(-128, 127)
W_dequant = W_int8 * scale
```
Apply it to the first block's `out_proj` and check reconstruction error.

**E11.3** (Research) Explain **GPTQ** and **AWQ** quantization methods. What makes them more accurate than naive round-to-nearest for large models?

**E11.4** Use `torch.export` or `torch.ao.quantization.quantize_pt2e` (PyTorch 2.0+ quantization flow) to apply static INT8 quantization. Compare size, speed and accuracy vs. dynamic quantization.


In [ ]:
# ── E11.1 — Selective quantization ───────────────────────────────────────────
# Quantize only FFN (PositionwiseFFN) linear layers
selective_q = torch.quantization.quantize_dynamic(
    fp32_model,
    {nn.Linear},          # YOUR CODE: filter to only FFN linears
    dtype=torch.qint8
)
print(f"Selective INT8 size : {model_size_mb(selective_q):.2f} MB")

# ── E11.2 — Absmax INT8 quantizer ────────────────────────────────────────────
def absmax_quant_dequant(weight: torch.Tensor):
    """Round-to-nearest INT8 with absmax scaling."""
    # YOUR CODE HERE
    scale = weight.abs().max() / 127.0
    w_int8 = weight.div(scale).round().clamp(-128, 127).to(torch.int8)
    w_dequant = w_int8.to(torch.float32) * scale
    return w_dequant, scale

# Test on first block's out_proj
w_orig = fp32_model.blocks[0].attn.out.weight.data.clone()
w_deq, scale = absmax_quant_dequant(w_orig)
err = (w_orig - w_deq).abs()
print(f"\nE11.2 absmax quant:")
print(f"  Scale             : {scale:.6f}")
print(f"  Mean abs error    : {err.mean():.6f}")
print(f"  Max  abs error    : {err.max():.6f}")
print(f"  Relative SNR (dB) : {20*torch.log10(w_orig.norm()/err.norm()):.2f} dB")

print("\nStubs loaded — implement E11.1–E11.4 above.")


## Section 12 — Advanced Optimization Recipes

### Techniques covered

| Recipe | What it does | Gain |
|---|---|---|
| Fused AdamW | Single kernel for param update | 10–20% step time |
| AMP + compile | Mixed precision inside JIT graph | 1.5–2× throughput |
| Gradient clipping | Stabilise large-batch training | Stability |
| Weight initialisation | Scaled init for deep networks | Faster convergence |
| Cosine + warmup schedule | Smooth LR decay | Better final loss |
| Gradient accumulation | Simulate large batches | Memory efficiency |

### Fused optimisers

PyTorch ships fused CUDA kernels for AdamW, SGD, and Adam:

```python
opt = torch.optim.AdamW(model.parameters(), fused=True)   # PyTorch ≥ 2.0, CUDA only
```

### Weight initialisation for Transformers

GPT-2 paper uses scaled init for residual projections:

```python
std = 0.02 / math.sqrt(2 * n_layers)
nn.init.normal_(proj_weight, mean=0.0, std=std)
```

### Warmup + cosine schedule

```python
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=3e-4, total_steps=STEPS,
    pct_start=0.05, anneal_strategy="cos"
)
```


In [ ]:
import math

# ── 1. Fused AdamW ────────────────────────────────────────────────────────────
def make_optimizer(model, lr=3e-4, weight_decay=0.1, use_fused=True):
    # Separate weight-decay and no-decay parameter groups (bias + LN params)
    decay     = {n for n, p in model.named_parameters() if p.dim() >= 2}
    no_decay  = {n for n, p in model.named_parameters() if p.dim() < 2}
    groups = [
        {"params": [p for n,p in model.named_parameters() if n in decay],
         "weight_decay": weight_decay},
        {"params": [p for n,p in model.named_parameters() if n in no_decay],
         "weight_decay": 0.0},
    ]
    fused_available = "fused" in torch.optim.AdamW.__init__.__code__.co_varnames
    use_fused = use_fused and fused_available and DEVICE == "cuda"
    return torch.optim.AdamW(groups, lr=lr,
                              fused=use_fused if fused_available else False)

# ── 2. Scaled weight init (GPT-2 style) ──────────────────────────────────────
def init_weights_gpt2(model, n_layers):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            std = 0.02
            if "out" in name:   # residual projection: scale by depth
                std = 0.02 / math.sqrt(2 * n_layers)
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

# ── 3. Full training recipe: AMP + compile + fused opt + cosine schedule ─────
STEPS        = 100
WARMUP_STEPS = 10
ACCUM_STEPS  = 4    # gradient accumulation factor

model = GPT(VOCAB_SIZE, D_MODEL, NUM_HEADS, N_LAYERS, D_FF, SEQ_LEN).to(DEVICE)
init_weights_gpt2(model, N_LAYERS)

opt    = make_optimizer(model)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")
sched  = torch.optim.lr_scheduler.OneCycleLR(
             opt, max_lr=3e-4,
             total_steps=STEPS // ACCUM_STEPS,
             pct_start=WARMUP_STEPS / STEPS,
             anneal_strategy="cos")

# Optional: compile
try:
    model = torch.compile(model, mode="default")
    print("torch.compile enabled ✓")
except Exception:
    print("torch.compile not available — running eager")

model.train()
opt.zero_grad()
running_loss = 0.0

for step in range(STEPS):
    ids    = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN), device=DEVICE)
    target = ids[:, 1:]
    inp    = ids[:, :-1]

    # AMP forward
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=DEVICE=="cuda"):
        logits = model(inp)
        loss   = F.cross_entropy(logits.view(-1, VOCAB_SIZE), target.reshape(-1))
        loss   = loss / ACCUM_STEPS

    scaler.scale(loss).backward()
    running_loss += loss.item()

    if (step + 1) % ACCUM_STEPS == 0:
        scaler.unscale_(opt)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0).item()
        scaler.step(opt); scaler.update()
        sched.step()
        opt.zero_grad()

        if (step + 1) % 20 == 0:
            avg_loss = running_loss * ACCUM_STEPS / 20
            lr_now   = sched.get_last_lr()[0]
            print(f"  step {step+1:3d} | loss {avg_loss:.4f} | "
                  f"grad_norm {grad_norm:.3f} | lr {lr_now:.2e}")
            running_loss = 0.0

print("\n=== Training recipe complete ✓ ===")
print("Components used: GPT-2 init | AdamW (weight-decay groups) | "
      "AMP | Gradient Accumulation | Cosine+Warmup LR | Gradient Clipping")

# ── 4. Gradient accumulation vs. large batch comparison ──────────────────────
print("\n=== Gradient accumulation equivalence check ===")
torch.manual_seed(0)
m1 = GPT(VOCAB_SIZE, D_MODEL, NUM_HEADS, 2, D_FF, SEQ_LEN).to(DEVICE)
m2 = GPT(VOCAB_SIZE, D_MODEL, NUM_HEADS, 2, D_FF, SEQ_LEN).to(DEVICE)
with torch.no_grad():
    for p1, p2 in zip(m1.parameters(), m2.parameters()):
        p2.copy_(p1)

torch.manual_seed(42)
ids_big = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE * 4, SEQ_LEN), device=DEVICE)

# Large batch in one step
logits_big = m1(ids_big[:, :-1])
loss_big   = F.cross_entropy(logits_big.view(-1, VOCAB_SIZE), ids_big[:, 1:].reshape(-1))
loss_big.backward()

# 4 accumulated micro-steps
for chunk in ids_big.chunk(4):
    logits_c = m2(chunk[:, :-1])
    loss_c   = F.cross_entropy(logits_c.view(-1, VOCAB_SIZE), chunk[:, 1:].reshape(-1)) / 4
    loss_c.backward()

# Check gradient match
max_diff = max((p1.grad - p2.grad).abs().max().item()
               for p1, p2 in zip(m1.parameters(), m2.parameters())
               if p1.grad is not None)
print(f"  Max grad diff (large-batch vs accumulated): {max_diff:.2e}  ✓")


### 🏋️ Exercises — Section 12: Advanced Optimization Recipes

**E12.1** Implement `make_optimizer` with **separate learning rates** for embeddings (smaller LR) and transformer blocks. Train two models — uniform LR vs. per-group LR — and compare convergence curves.

**E12.2** Replace the `OneCycleLR` scheduler with a **linear warmup + cosine decay** scheduler implemented from scratch using `LambdaLR`. Verify the LR trace matches the expected curve analytically.

**E12.3** Implement **Lion optimizer** from scratch (sign-based update rule):
```
m_t = β1*m_{t-1} + (1-β1)*g_t
θ_t = θ_{t-1} - lr * sign(m_t + weight_decay*θ_{t-1})
```
Compare convergence speed and final loss vs. AdamW on 200 steps.

**E12.4** Run a hyperparameter sweep over `{lr, dropout, weight_decay}` using a simple grid search. Log all runs with `writer.add_hparams` and identify the best configuration by validation loss.


In [ ]:
# ── E12.1 — Per-group learning rates ─────────────────────────────────────────
def make_optimizer_per_group(model, base_lr=3e-4, embed_lr_factor=0.1, weight_decay=0.1):
    """Different learning rates for embeddings vs. transformer body."""
    embed_params  = list(model.tok_emb.parameters()) + list(model.pos_emb.parameters())
    embed_names   = {id(p) for p in embed_params}
    body_params   = [p for p in model.parameters() if id(p) not in embed_names and p.dim() >= 2]
    nodecay_params= [p for p in model.parameters() if id(p) not in embed_names and p.dim() < 2]
    groups = [
        # YOUR CODE HERE: add weight_decay and different lr for each group
        {"params": embed_params,   "lr": base_lr * embed_lr_factor, "weight_decay": 0.0},
        {"params": body_params,    "lr": base_lr,                   "weight_decay": weight_decay},
        {"params": nodecay_params, "lr": base_lr,                   "weight_decay": 0.0},
    ]
    return torch.optim.AdamW(groups)

opt_pg = make_optimizer_per_group(model)
print("Per-group optimizer param groups:")
for i, g in enumerate(opt_pg.param_groups):
    n = sum(p.numel() for p in g["params"])
    print(f"  group {i}: {n:,} params | lr={g['lr']:.2e} | wd={g['weight_decay']}")

# ── E12.2 — Linear warmup + cosine decay from scratch ────────────────────────
def warmup_cosine_lambda(step, warmup_steps, total_steps, min_lr_ratio=0.1):
    """Returns LR multiplier for LambdaLR."""
    # YOUR CODE HERE:
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return min_lr_ratio + (1 - min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * progress))

opt_wc = torch.optim.AdamW(model.parameters(), lr=3e-4)
sched_wc = torch.optim.lr_scheduler.LambdaLR(
    opt_wc, lr_lambda=lambda s: warmup_cosine_lambda(s, 10, 100))
lrs = [sched_wc.get_last_lr()[0]]
for _ in range(99): sched_wc.step(); lrs.append(sched_wc.get_last_lr()[0])
print(f"\nE12.2 warmup-cosine LR trace (first 10): {[f'{v:.4f}' for v in lrs[:10]]}")

# ── E12.3 — Lion Optimizer ────────────────────────────────────────────────────
class Lion(torch.optim.Optimizer):
    """Lion optimizer (Chen et al., 2023)."""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure else None
        for group in self.param_groups:
            lr, wd = group["lr"], group["weight_decay"]
            b1, b2 = group["betas"]
            for p in group["params"]:
                if p.grad is None: continue
                g = p.grad
                state = self.state[p]
                if "m" not in state:
                    state["m"] = torch.zeros_like(p)
                m = state["m"]
                # YOUR CODE HERE:
                # update = sign(b1*m + (1-b1)*g)
                # p  -= lr * (update + wd * p)
                # m   = b2*m + (1-b2)*g
                update = (b1 * m + (1 - b1) * g).sign()
                p.add_(update + wd * p, alpha=-lr)
                m.mul_(b2).add_(g, alpha=1 - b2)
        return loss

lion = Lion(model.parameters(), lr=1e-4, weight_decay=0.1)
print("\nE12.3 Lion optimizer created ✓")

# ── E12.4 — Hyperparameter sweep stub ────────────────────────────────────────
# YOUR CODE HERE:
# import itertools
# for lr, dropout, wd in itertools.product([1e-3, 3e-4], [0.0, 0.1], [0.0, 0.1]):
#     ...train and log with writer.add_hparams(...)
print("E12.4: implement the hyperparameter sweep above.")
print("\n✅ pytorch_advanced.ipynb complete! All 12 sections with exercises.")
